In [ ]:
!pip install biopython
!pip install py3Dmol


!pip install vina --quiet
!pip install meeko --quiet
!pip install openbabel-wheel --quiet
!pip install biopython py3Dmol --quiet

print("Installation complete")

Installation complete


In [ ]:
!pip install vina meeko --force-reinstall --quiet
!apt-get install -y openbabel --quiet

# Check Vina
try:
    from vina import Vina
    print("Vina: OK")
except ImportError as e:
    print(f"Vina: FAILED — {e}")

# Check Open Babel (your Meeko replacement)
import subprocess
result = subprocess.run(["obabel", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"Open Babel: OK — {result.stdout.split()[2] if len(result.stdout.split()) > 2 else 'installed'}")
else:
    print("Open Babel: FAILED")

# Check RDKit (backup for molecule handling if needed)
try:
    from rdkit import Chem
    print("RDKit: OK")
except ImportError:
    print("RDKit: not available (not critical)")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
Reading package lists...
Building dependency tree...
Reading state information...
openbabel is already the newest version (3.1.1+dfsg-6ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Vina: OK
Open Babel: FAILED
RDKit: not available (not critical)


In [ ]:
import subprocess

# Find where obabel actually lives after installation
result = subprocess.run(["find", "/usr", "-name", "obabel", "-type", "f"],
                         capture_output=True, text=True)
print("obabel found at:", result.stdout.strip())

obabel found at: /usr/bin/obabel
/usr/local/bin/obabel
/usr/local/lib/python3.12/dist-packages/openbabel/bin/obabel


In [ ]:
from Bio import PDB
import urllib.request

# Download TphC structure directly from PDB
# Replace XXXX with your actual PDB ID
pdb_id = "7NDS"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
urllib.request.urlretrieve(url, f"{pdb_id}.pdb")

# Parse the structure
parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure(pdb_id, f"{pdb_id}.pdb")

# Print basic info
for model in structure:
    for chain in model:
        residues = list(chain.get_residues())
        print(f"Chain {chain.id}: {len(residues)} residues")

Chain A: 425 residues


In [ ]:
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa

def extract_sequence_and_mapping(pdb_file, chain_id="A"):
    """
    Extract sequence AND build a mapping from crystal residue number
    to sequence position. Crystal structures often have non-sequential
    or offset numbering (missing loops, cleaved signal peptides, etc.),
    while ESMFold predictions are numbered sequentially from 1 based
    purely on the submitted sequence. This mapping bridges the two.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    AA_3TO1 = {
        'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
        'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
        'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
        'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
    }

    sequence = ""
    crystal_to_seq_index = {}  # crystal residue number -> 0-indexed sequence position

    for model in structure:
        for chain in model:
            if chain.id == chain_id:
                for residue in chain:
                    resname = residue.get_resname().upper()
                    if resname in AA_3TO1:
                        crystal_res_num = residue.get_id()[1]
                        crystal_to_seq_index[crystal_res_num] = len(sequence)
                        sequence += AA_3TO1[resname]
        break

    return sequence, crystal_to_seq_index

# Re-run this to get both the sequence and the mapping
tphc_sequence, crystal_to_seq_index = extract_sequence_and_mapping(f"{pdb_id}.pdb", chain_id="A")
print(f"Sequence length: {len(tphc_sequence)}")
print(f"Mapping built for {len(crystal_to_seq_index)} residues")

Sequence length: 294
Mapping built for 294 residues


In [ ]:
import requests
import time

def predict_structure_esmfold(sequence, max_length=400, timeout=300):

    # Clean the sequence
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    sequence = ''.join([aa for aa in sequence.upper() if aa in valid_aa])

    # Truncate if too long
    if len(sequence) > max_length:
        print(f"Warning: sequence truncated from {len(sequence)} to {max_length} residues")
        sequence = sequence[:max_length]

    print(f"Sending sequence of length {len(sequence)} to ESMFold API...")

    try:
        response = requests.post(
            'https://api.esmatlas.com/foldSequence/v1/pdb/',
            headers={'Content-Type': 'application/x-www-form-urlencoded'},
            data=sequence.encode('utf-8'),
            timeout=timeout
        )

        # Check HTTP status
        if response.status_code != 200:
            print(f"API returned status code: {response.status_code}")
            print(f"Response content: {response.text[:500]}")
            return None

        # Check if response looks like a PDB file
        if "ATOM" not in response.text:
            print("Response received but doesn't contain structure data:")
            print(response.text[:500])
            return None

        print("Structure prediction successful")
        return response.text

    except requests.exceptions.Timeout:
        print("Request timed out — try increasing timeout or using a shorter sequence")
        return None

    except requests.exceptions.ConnectionError:
        print("Connection error — check your internet connection or try again later")
        return None

    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

In [ ]:
import py3Dmol

def visualize_pdb(pdb_string):
    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_string, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    view.show()
    return view

print("Attempting to predict structure...")
result = None # Initialize result to ensure it's always defined
try:
    result = predict_structure_esmfold(tphc_sequence)
    print(f"predict_structure_esmfold returned: {type(result)} {str(result)[:50]}...")
except Exception as e:
    print(f"Error during predict_structure_esmfold: {e}")
    result = None # Ensure result is still None on error

# If ESMFold worked, visualize the result
if result and "ATOM" in result:
    print("Visualizing PDB structure.")
    visualize_pdb(result)
else:
    print("Could not visualize PDB structure. 'result' was empty or did not contain 'ATOM' keyword.")
    if result is None:
        print("Reason: 'result' is None.")
    elif not result:
        print("Reason: 'result' is an empty string or Falsey.")
    elif "ATOM" not in result:
        print("Reason: 'ATOM' not found in 'result'.")
    else:
        print("Reason: Unknown issue.")

Attempting to predict structure...
Sending sequence of length 294 to ESMFold API...
Structure prediction successful
predict_structure_esmfold returned: <class 'str'> HEADER                                            ...
Visualizing PDB structure.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# Run predictions and saving results
print(f"Sequence length: {len(tphc_sequence)} residues")
print(f"First 50 residues: {tphc_sequence[:50]}")

# Run ESMFold prediction
print("\nSending to ESMFold API... this may take 1-3 minutes")
tphc_predicted_pdb = predict_structure_esmfold(tphc_sequence)

# Check if it worked
if tphc_predicted_pdb is None:
    print("Prediction failed — see error message above")
else:
    print(f"Success! Received {len(tphc_predicted_pdb)} characters of PDB data")

    # Save to file
    with open("tphc_predicted.pdb", "w") as f:
        f.write(tphc_predicted_pdb)
    print("Saved to tphc_predicted.pdb")

Sequence length: 294 residues
First 50 residues: SNQPLKIVVPFSAGGTADVLPRLVAEKIRADYAGGVIIENKPGAGGNIGA

Sending to ESMFold API... this may take 1-3 minutes
Sending sequence of length 294 to ESMFold API...
Structure prediction successful
Success! Received 181239 characters of PDB data
Saved to tphc_predicted.pdb


In [ ]:

def visualize_single_structure(pdb_string, title="Structure",
                                style="cartoon", color="spectrum"):
    print(f"Visualizing: {title}")

    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_string, 'pdb')
    view.setStyle({style: {'color': color}})
    view.addLabel(title,
                  {'position': {'x': 0, 'y': 0, 'z': 0},
                   'fontSize': 14,
                   'fontColor': 'white'})
    view.zoomTo()
    view.show()
    return view  # restored — this is actually what produces the render



In [ ]:
# Load the crystal structure from the file you downloaded on Day 2
pdb_id = "7NDS"

with open(f"{pdb_id}.pdb", "r") as f:
    crystal_pdb_string = f.read()

visualize_single_structure(crystal_pdb_string,
                           title="TphC - Crystal Structure",
                           color="cyan")

Visualizing: TphC - Crystal Structure


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
from Bio.PDB import PDBParser, NeighborSearch

STANDARD_AA = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
               'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'}

def get_binding_pocket_residues(pdb_file, ligand_name="UB7", radius=5.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    all_atoms = list(structure.get_atoms())

    ligand_atoms = [atom for atom in all_atoms
                    if atom.get_parent().get_resname() == ligand_name]

    if not ligand_atoms:
        print(f"Warning: ligand '{ligand_name}' not found in {pdb_file}")
        return []

    print(f"Found {len(ligand_atoms)} atoms in ligand {ligand_name}")

    ns = NeighborSearch(all_atoms)
    nearby_residues = set()

    for ligand_atom in ligand_atoms:
        neighbors = ns.search(ligand_atom.get_vector().get_array(),
                             radius, 'R')
        for residue in neighbors:
            resname = residue.get_resname()
            # Only keep standard amino acids — excludes ligand, water,
            # and any other crystallographic additives automatically
            if resname in STANDARD_AA:
                res_id = residue.get_id()[1]
                nearby_residues.add((res_id, resname))

    pocket_residues = sorted(nearby_residues, key=lambda x: x[0])

    print(f"\nBinding pocket residues (amino acids only) within {radius}Å of {ligand_name}:")
    print("-" * 40)
    for res_id, res_name in pocket_residues:
        print(f"  Residue {res_id}: {res_name}")

    return pocket_residues

# Re-run with the corrected function
pocket_residues = get_binding_pocket_residues(f"{pdb_id}.pdb", ligand_name="UB7")

Found 16 atoms in ligand UB7

Binding pocket residues (amino acids only) within 5.0Å of UB7:
----------------------------------------
  Residue 38: PHE
  Residue 39: SER
  Residue 42: GLY
  Residue 43: THR
  Residue 44: ALA
  Residue 45: ASP
  Residue 94: PRO
  Residue 95: PRO
  Residue 96: GLY
  Residue 126: ASN
  Residue 158: GLN
  Residue 162: SER
  Residue 163: THR
  Residue 164: SER
  Residue 185: LYS
  Residue 186: GLY
  Residue 187: THR
  Residue 188: ALA
  Residue 205: ASN
  Residue 250: THR
  Residue 252: PHE


In [ ]:
def visualize_binding_pocket(pdb_string, ligand_name="UB7", radius=8.0):
    """
    Visualize just the binding pocket with the ligand highlighted
    """
    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_string, 'pdb')

    # Show the whole protein as a faint cartoon
    view.setStyle({'cartoon': {'color': 'lightgrey', 'opacity': 0.5}})

    # Show the ligand prominently
    view.addStyle({'resn': ligand_name},
                  {'stick': {'color': 'yellow', 'radius': 0.3}})
    view.addStyle({'resn': ligand_name},
                  {'sphere': {'color': 'yellow', 'opacity': 0.3, 'radius': 1.5}})

    # Show pocket residues as colored sticks
    view.addStyle({'within': {'distance': radius, 'sel': {'resn': ligand_name}}},
                  {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.2}})

    # Zoom to the ligand
    view.zoomTo({'resn': ligand_name})
    view.show()

visualize_binding_pocket(crystal_pdb_string, ligand_name="UB7")

print(f"'UB7' appears in crystal_pdb_string: {'UB7' in crystal_pdb_string}")

# Count how many lines reference UB7
ub7_lines = [line for line in crystal_pdb_string.split('\n') if 'UB7' in line]
print(f"Number of lines containing UB7: {len(ub7_lines)}")
if ub7_lines:
    print("Sample line:", ub7_lines[0])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

'UB7' appears in crystal_pdb_string: True
Number of lines containing UB7: 21
Sample line: REMARK 800 SITE_DESCRIPTION: binding site for residue UB7 A 401                 


In [ ]:
from Bio.PDB import PDBParser, NeighborSearch, Superimposer
import io

def calculate_pocket_rmsd(crystal_file, predicted_pdb_string,
                           pocket_residue_ids, crystal_to_seq_index,
                           chain_id="A"):
    parser = PDBParser(QUIET=True)

    crystal_struct = parser.get_structure("crystal", crystal_file)
    predicted_struct = parser.get_structure("predicted", io.StringIO(predicted_pdb_string))

    crystal_pocket_nums = [res_id for res_id, res_name in pocket_residue_ids]

    # Convert crystal residue numbers -> predicted structure residue numbers
    # via sequence position (predicted numbering = seq_index + 1)

    predicted_pocket_nums = []
    for crystal_num in crystal_pocket_nums:
        if crystal_num in crystal_to_seq_index:
            seq_idx = crystal_to_seq_index[crystal_num]
            predicted_pocket_nums.append(seq_idx + 1)
        else:
            print(f"Warning: crystal residue {crystal_num} has no sequence mapping — skipping")

    def get_ca_by_number(structure, chain_id, residue_numbers):
        residue_numbers = set(residue_numbers)
        ca_atoms = []
        for model in structure:
            for chain in model:
                if chain.id == chain_id:
                    for residue in chain:
                        res_num = residue.get_id()[1]
                        if res_num in residue_numbers and 'CA' in residue:
                            ca_atoms.append((res_num, residue['CA']))
            break
        ca_atoms.sort(key=lambda x: x[0])
        return ca_atoms

    crystal_ca = get_ca_by_number(crystal_struct, chain_id, crystal_pocket_nums)
    predicted_ca = get_ca_by_number(predicted_struct, "A", predicted_pocket_nums)

    print(f"Crystal pocket CA atoms matched: {len(crystal_ca)}")
    print(f"Predicted pocket CA atoms matched: {len(predicted_ca)}")

    min_len = min(len(crystal_ca), len(predicted_ca))

    if min_len < 3:
        print("Still too few matched residues — see troubleshooting notes below")
        return None

    crystal_atoms_only = [atom for _, atom in crystal_ca[:min_len]]
    predicted_atoms_only = [atom for _, atom in predicted_ca[:min_len]]

    superimposer = Superimposer()
    superimposer.set_atoms(crystal_atoms_only, predicted_atoms_only)
    superimposer.apply(predicted_struct.get_atoms())

    pocket_rmsd = superimposer.rms
    print(f"Binding pocket RMSD: {pocket_rmsd:.3f} Å")
    return pocket_rmsd

# Run it with the mapping
pocket_rmsd = calculate_pocket_rmsd(f"{pdb_id}.pdb", tphc_predicted_pdb,
                                     pocket_residues, crystal_to_seq_index)

# Build predicted_res_nums for the range-check diagnostic
parser_check = PDBParser(QUIET=True)
predicted_struct_check = parser_check.get_structure("predicted_check", io.StringIO(tphc_predicted_pdb))
predicted_res_nums = []
for model in predicted_struct_check:
    for chain in model:
        if chain.id == "A":
            for residue in chain:
                if 'CA' in residue:
                    predicted_res_nums.append(residue.get_id()[1])
    break

print(f"Predicted structure residue number range: {min(predicted_res_nums)} to {max(predicted_res_nums)}")
print(f"Expected range if 1-indexed, no gaps: 1 to {len(tphc_sequence)}")

Crystal pocket CA atoms matched: 21
Predicted pocket CA atoms matched: 21
Binding pocket RMSD: 3.679 Å
Predicted structure residue number range: 1 to 294
Expected range if 1-indexed, no gaps: 1 to 294


In [ ]:
from Bio.PDB import PDBParser, Superimposer
import numpy as np
import io


STANDARD_AA = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
               'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'}

def per_residue_deviation(crystal_ca, predicted_ca):
    deviations = []
    for c_atom, p_atom in zip(crystal_ca, predicted_ca):
        c_coord = c_atom.get_coord()
        p_coord = p_atom.get_coord()
        dist = np.linalg.norm(c_coord - p_coord)
        deviations.append(dist)
    return deviations

def calculate_rmsd(crystal_file, predicted_pdb_string, chain_id="A"):
    """
    Calculate RMSD between crystal structure and ESMFold prediction.

    crystal_file: path to crystal structure PDB file
    predicted_pdb_string: ESMFold prediction as string
    chain_id: which chain to compare (usually "A")
    """
    parser = PDBParser(QUIET=True)

    crystal_struct = parser.get_structure("crystal", crystal_file)
    predicted_struct = parser.get_structure(
        "predicted",
        io.StringIO(predicted_pdb_string)
    )

    def get_ca_atoms(structure, chain_id):
        ca_atoms = []
        for model in structure:
            for chain in model:
                if chain.id == chain_id:
                    for residue in chain:
                        if residue.get_resname() in STANDARD_AA and 'CA' in residue:
                            ca_atoms.append(residue['CA'])
            break  # only first model
        return ca_atoms

    crystal_ca = get_ca_atoms(crystal_struct, chain_id)
    predicted_ca = get_ca_atoms(predicted_struct, "A")  # ESMFold always uses chain A

    print(f"Crystal structure CA atoms: {len(crystal_ca)}")
    print(f"Predicted structure CA atoms: {len(predicted_ca)}")

    min_length = min(len(crystal_ca), len(predicted_ca))
    crystal_ca = crystal_ca[:min_length]
    predicted_ca = predicted_ca[:min_length]

    if min_length < 10:
        print("Error: too few atoms to calculate meaningful RMSD")
        return None

    superimposer = Superimposer()
    superimposer.set_atoms(crystal_ca, predicted_ca)
    superimposer.apply(predicted_struct.get_atoms())   # mutates predicted_ca's coordinates in place

    deviations = per_residue_deviation(crystal_ca, predicted_ca)

    rmsd = superimposer.rms

    print(f"\nRMSD (Cα atoms, {min_length} residues): {rmsd:.3f} Å")

    print(f"Mean per-residue deviation: {np.mean(deviations):.2f} Å")
    print(f"Max per-residue deviation: {np.max(deviations):.2f} Å "
          f"at position {np.argmax(deviations) + 1}")

    high_dev_positions = [(i + 1, d) for i, d in enumerate(deviations) if d > 5.0]
    if high_dev_positions:
        print(f"\nPositions with deviation > 5.0 Å ({len(high_dev_positions)} total):")
        for pos, dev in high_dev_positions:
            print(f"  Position {pos}: {dev:.2f} Å")
    else:
        print("\nNo positions exceed 5.0 Å deviation.")

    print("\nInterpretation:")
    if rmsd < 1.0:
        print("  Excellent — near-identical structures, high confidence in predictions")
    elif rmsd < 2.0:
        print("  Very good — minor differences likely in loop regions, predictions reliable")
    elif rmsd < 3.0:
        print("  Good — overall fold is correct, some local differences exist")
    elif rmsd < 4.0:
        print("  Moderate — fold is approximately correct but treat predictions cautiously")
    else:
        print("  Poor — significant structural differences, verify your sequence and structure")

    return rmsd, deviations

print("Binding pocket residues — predicted-structure deviation check:")
print("-" * 65)

pocket_seq_positions = []
for crystal_num, res_name in pocket_residues:
    if crystal_num in crystal_to_seq_index:
        seq_pos = crystal_to_seq_index[crystal_num] + 1  # 1-indexed to match 'position' numbering above
        pocket_seq_positions.append((crystal_num, res_name, seq_pos))
    else:
        print(f"  Crystal residue {crystal_num} ({res_name}) — no sequence mapping, skipping")

flagged_count = 0
for crystal_num, res_name, seq_pos in pocket_seq_positions:
    if seq_pos <= len(deviations):
        dev = deviations[seq_pos - 1]  # deviations list is 0-indexed
        flag = "  <-- HIGH DEVIATION" if dev > 5.0 else ""
        if dev > 5.0:
            flagged_count += 1
        print(f"  Crystal residue {crystal_num} ({res_name}) -> seq position {seq_pos}: {dev:.2f} Å{flag}")

print(f"\n{flagged_count} of {len(pocket_seq_positions)} pocket residues exceed 5.0 Å deviation")

# Run the comparison
rmsd_value, deviations = calculate_rmsd(f"{pdb_id}.pdb", tphc_predicted_pdb)

Binding pocket residues — predicted-structure deviation check:
-----------------------------------------------------------------


NameError: name 'deviations' is not defined

In [ ]:
def visualize_comparison(crystal_pdb_string, predicted_pdb_string):
    """
    Show crystal and predicted structures overlaid for visual comparison
    """
    view = py3Dmol.view(width=900, height=600)

    # Add crystal structure in cyan
    view.addModel(crystal_pdb_string, 'pdb')
    view.setStyle({'model': 0},
                  {'cartoon': {'color': 'cyan', 'opacity': 0.8}})

    # Add predicted structure in orange
    view.addModel(predicted_pdb_string, 'pdb')
    view.setStyle({'model': 1},
                  {'cartoon': {'color': 'orange', 'opacity': 0.8}})

    view.zoomTo()
    view.show()

    print("Cyan = Crystal structure")
    print("Orange = ESMFold prediction")
    print("Regions where they overlap closely = well-predicted areas")
    print("Regions where they diverge = less reliable in predictions")

visualize_comparison(crystal_pdb_string, tphc_predicted_pdb)

In [ ]:
print("=" * 60)
print("DAY 4 SUMMARY")
print("=" * 60)
print(f"\nProtein: TphC (PDB ID: {pdb_id})")
print(f"Sequence length: {len(tphc_sequence)} residues")
print(f"\nBinding pocket residues identified: {len(pocket_residues)}")
for res_id, res_name in pocket_residues:
    print(f"  {res_name} {res_id}")
print(f"\nWhole-protein RMSD (crystal vs ESMFold): {rmsd_value:.3f} Å")
print(f"Binding pocket RMSD: {pocket_rmsd:.3f} Å")
print("\nConclusion for pipeline:")

if pocket_rmsd < 2.0:
    print("  ESMFold predictions are reliable for this protein family.")
    print("  Proceed with confidence using predicted structures for variant design.")
elif pocket_rmsd < 3.0:
    print("  ESMFold predictions are approximately reliable.")
    print("  Use crystal structure for docking; use predictions for qualitative comparisons.")
else:
    print("  ESMFold predictions show significant deviation in the pocket.")
    print("  Rely primarily on crystal structure for all docking work.")

In [ ]:
print("Pocket residues (crystal numbering):")
print(pocket_residues)

# Check the residue number range in the crystal structure
parser = PDBParser(QUIET=True)
crystal_struct = parser.get_structure("crystal", f"{pdb_id}.pdb")
crystal_res_nums = []
for model in crystal_struct:
    for chain in model:
        if chain.id == "A":
            for residue in chain:
                if 'CA' in residue:
                    crystal_res_nums.append(residue.get_id()[1])
    break

print(f"\nCrystal structure residue numbers range: {min(crystal_res_nums)} to {max(crystal_res_nums)}")
print(f"Total CA atoms in crystal: {len(crystal_res_nums)}")

# Check the residue number range in the predicted structure
import io
predicted_struct = parser.get_structure("predicted", io.StringIO(tphc_predicted_pdb))
predicted_res_nums = []
for model in predicted_struct:
    for chain in model:
        if chain.id == "A":
            for residue in chain:
                if 'CA' in residue:
                    predicted_res_nums.append(residue.get_id()[1])
    break

print(f"Predicted structure residue numbers range: {min(predicted_res_nums)} to {max(predicted_res_nums)}")
print(f"Total CA atoms in predicted: {len(predicted_res_nums)}")